**Assignment: Fine-Tuning BERT for POS Tagging & Chunking**



**Objective**

Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.




STEP 1: Install Libraries

In [2]:
!pip install -q transformers datasets evaluate seqeval accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


STEP 2: Import Libraries

In [3]:
import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed
)
from pprint import pprint


STEP 3: Setup Device & Seed

In [4]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


STEP 4: Load Dataset (POS Tagging)

In [5]:
from datasets import load_dataset

for name in [
    "Babelscape/conll2003-nemo",
    "conll2003",
    "wikiann",
]:
    try:
        if name == "wikiann":
            dataset = load_dataset("wikiann", "en")
        else:
            dataset = load_dataset(name)

        print(f"✅ Loaded dataset: {name}")
        break

    except Exception as e:
        print(f"⚠️ Could not load {name}: {e}")

else:
    raise RuntimeError("❌ Unable to load any dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


⚠️ Could not load Babelscape/conll2003-nemo: Dataset 'Babelscape/conll2003-nemo' doesn't exist on the Hub or cannot be accessed.


README.md: 0.00B [00:00, ?B/s]

conll2003.py: 0.00B [00:00, ?B/s]

⚠️ Could not load conll2003: Dataset scripts are no longer supported, but found conll2003.py


...

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

✅ Loaded dataset: wikiann


STEP 6: Load Tokenizer

In [6]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

STEP 5: Identify Labels

In [7]:
label_col = "ner_tags"

label_list = dataset["train"].features[label_col].feature.names
num_labels = len(label_list)

print("Labels:", label_list)

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [16]:
small_train = dataset["train"].select(range(2000))
small_val = dataset["validation"].select(range(500))

STEP 7: Tokenization + Label Alignment

In [17]:
def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        word_labels = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                word_labels.append(-100)
            elif word_idx != previous_word_idx:
                word_labels.append(examples[label_col][i][word_idx])
            else:
                word_labels.append(-100)

            previous_word_idx = word_idx

        labels.append(word_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

small_train = small_train.map(align_labels_with_tokens, batched=True)
small_val = small_val.map(align_labels_with_tokens, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [18]:
sample = small_train[0]

print("Input IDs:", sample["input_ids"])
print("Attention Mask:", sample["attention_mask"])
print("Labels:", sample["labels"])

Input IDs: [101, 1054, 1012, 1044, 1012, 15247, 1006, 2358, 1012, 5623, 2314, 1007, 1006, 5986, 2620, 12464, 1007, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels: [-100, 3, -100, -100, -100, 4, 0, 3, -100, 4, 4, 0, 0, 0, -100, 0, 0, -100]


STEP 8: Data Collator

In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer)

STEP 9: Load Model

In [20]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

STEP 10: Evaluation Metrics


In [21]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

STEP 11: Training Arguments

In [22]:
args = TrainingArguments(
    output_dir="ner_model",
    eval_strategy="epoch",   # your version
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,      # FAST
    max_steps=500,           # LIMIT
    weight_decay=0.01,
)

STEP 12: Trainer Setup

In [23]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

STEP 13: Train Model

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.363194,0.594560,0.685075,0.636616,0.881485
2,0.544104,0.312488,0.692935,0.761194,0.725462,0.903611


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=500, training_loss=0.544103759765625, metrics={'train_runtime': 463.0281, 'train_samples_per_second': 8.639, 'train_steps_per_second': 1.08, 'total_flos': 28221579325104.0, 'train_loss': 0.544103759765625, 'epoch': 2.0})

STEP 14: Evaluate Model

In [25]:
trainer.evaluate()

{'eval_loss': 0.3124881088733673,
 'eval_precision': 0.6929347826086957,
 'eval_recall': 0.7611940298507462,
 'eval_f1': 0.7254623044096729,
 'eval_accuracy': 0.9036113936927772,
 'eval_runtime': 13.4866,
 'eval_samples_per_second': 37.074,
 'eval_steps_per_second': 4.671,
 'epoch': 2.0}


STEP 15: Inference

In [28]:
nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

text = "Ram got Data Analyst job ,John works at Google in California"
output = nlp(text)

for token in output:
    print(token)

{'entity': 'B-PER', 'score': np.float32(0.4441643), 'index': 1, 'word': 'ram', 'start': 0, 'end': 3}
{'entity': 'I-ORG', 'score': np.float32(0.7164657), 'index': 3, 'word': 'data', 'start': 8, 'end': 12}
{'entity': 'I-ORG', 'score': np.float32(0.64138997), 'index': 4, 'word': 'analyst', 'start': 13, 'end': 20}
{'entity': 'B-PER', 'score': np.float32(0.6548043), 'index': 7, 'word': 'john', 'start': 26, 'end': 30}
{'entity': 'I-PER', 'score': np.float32(0.49771506), 'index': 8, 'word': 'works', 'start': 31, 'end': 36}
{'entity': 'B-ORG', 'score': np.float32(0.6797176), 'index': 10, 'word': 'google', 'start': 40, 'end': 46}
{'entity': 'B-LOC', 'score': np.float32(0.39154226), 'index': 12, 'word': 'california', 'start': 50, 'end': 60}
